# 02 — Feature Engineering

> **Project**: CS5483 ChallengeData #163 — Predict Parking Violations  
> **Phase 2**: Transform 10 raw features into ~25-30 engineered features  
> **Tier System**: Tier 1 (must-do) → Tier 2 (should-do)  

Key principles:
1. Train and test feature pipelines must be consistent
2. Target Encoding uses K-Fold on train / full-train stats on test (no leakage)
3. Save parquet checkpoints after each tier
4. Use `gc.collect()` to free intermediate memory

In [1]:
# === Setup ===
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.model_selection import KFold
import pickle
import gc
import warnings

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DATA_DIR = '../163-Predict parking violations/'
OUTPUT_DIR = '../data/'

print('Libraries loaded')

Libraries loaded


In [2]:
# === Data Loading (memory-efficient) ===
dtype_dict = {
    'total_count': 'int32',
    'longitude_scaled': 'float32',
    'latitude_scaled': 'float32',
    'Precipitations': 'float32',
    'HauteurNeige': 'float32',
    'Temperature': 'float32',
    'ForceVent': 'float32',
    'day_of_week': 'int8',
    'month_of_year': 'int8',
    'hour': 'int8',
}

x_train = pd.read_csv(f'{DATA_DIR}x_train_final_asAbTs5.csv', dtype=dtype_dict, index_col=0)
y_train = pd.read_csv(f'{DATA_DIR}y_train_final_YYyFil7.csv', index_col=0)
x_test = pd.read_csv(f'{DATA_DIR}x_test_final_fIrnA7Q.csv', dtype=dtype_dict, index_col=0)

train_df = x_train.copy()
train_df['invalid_ratio'] = y_train['invalid_ratio'].astype('float32')
test_df = x_test.copy()

print(f'Training set: {train_df.shape[0]:,} rows, {train_df.shape[1]} columns')
print(f'Test set:     {test_df.shape[0]:,} rows, {test_df.shape[1]} columns')
print(f'Memory: train_df = {train_df.memory_usage(deep=True).sum()/1024**2:.0f} MB, '
      f'test_df = {test_df.memory_usage(deep=True).sum()/1024**2:.0f} MB')

del x_train, y_train, x_test
gc.collect()

Training set: 6,076,546 rows, 11 columns
Test set:     2,028,750 rows, 10 columns
Memory: train_df = 203 MB, test_df = 60 MB


0

---
## Tier 1: Must-Do Features

These features alone should significantly beat the baseline (Spearman 0.197).

### T1.1 Missing Value Imputation

Only two features have missing values:
- `HauteurNeige` (snow depth, 2.7%) → fill with 0 (no snow is a reasonable default)
- `ForceVent` (wind force, 0.1%) → fill with training set median

In [3]:
# === T1.1: Missing Value Imputation ===
# Save training set median BEFORE filling (for test set consistency)
forcevent_median = float(train_df['ForceVent'].median())
print(f'ForceVent training median: {forcevent_median}')

# Fill training set
train_df['HauteurNeige'] = train_df['HauteurNeige'].fillna(0)
train_df['ForceVent'] = train_df['ForceVent'].fillna(forcevent_median)

# Fill test set (using training set median!)
test_df['HauteurNeige'] = test_df['HauteurNeige'].fillna(0)
test_df['ForceVent'] = test_df['ForceVent'].fillna(forcevent_median)

# Verify no missing values remain
assert train_df.isnull().sum().sum() == 0, 'Train still has NaN!'
assert test_df.isnull().sum().sum() == 0, 'Test still has NaN!'
print('Missing value imputation complete. No NaN remaining.')

ForceVent training median: 3.200000047683716


Missing value imputation complete. No NaN remaining.


### T1.2 total_count Transformations

- **Log transform**: compress extreme values, stabilize variance  
- **Binning**: match the step-function relationship between count and violation rate

In [4]:
# === T1.2: total_count Transformations ===
# Log transform
train_df['log_total_count'] = np.log1p(train_df['total_count']).astype('float32')
test_df['log_total_count'] = np.log1p(test_df['total_count']).astype('float32')

# Binning (bins from EDA analysis)
count_bins = [0, 1, 3, 10, 30, np.inf]
count_labels = [0, 1, 2, 3, 4]

train_df['count_bin'] = pd.cut(
    train_df['total_count'], bins=count_bins, labels=count_labels
).astype('int8')
test_df['count_bin'] = pd.cut(
    test_df['total_count'], bins=count_bins, labels=count_labels
).astype('int8')

print('total_count transformations:')
print(f'  log_total_count range: [{train_df["log_total_count"].min():.2f}, {train_df["log_total_count"].max():.2f}]')
print(f'  count_bin distribution:')
print(train_df['count_bin'].value_counts().sort_index().to_string())

total_count transformations:
  log_total_count range: [0.69, 7.42]
  count_bin distribution:
count_bin
0    1532442
1    1150241
2    1751905
3    1343611
4     298347


### T1.3 Sin/Cos Cyclical Encoding

Temporal features (hour, day_of_week, month) are cyclical — encode with sin/cos  
to preserve the circular relationship (e.g., hour 19 is close to hour 6 the next day).

In [5]:
# === T1.3: Sin/Cos Cyclical Encoding ===
def add_cyclical_features(df):
    """Add sin/cos encoding for cyclical temporal features."""
    # Hour (period=24, preserves semantic even though data is 6-19)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24).astype('float32')
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24).astype('float32')
    
    # Day of week (period=7)
    df['dow_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7).astype('float32')
    df['dow_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7).astype('float32')
    
    # Month (period=12, shift by 1 so Jan=0)
    df['month_sin'] = np.sin(2 * np.pi * (df['month_of_year'] - 1) / 12).astype('float32')
    df['month_cos'] = np.cos(2 * np.pi * (df['month_of_year'] - 1) / 12).astype('float32')
    
    return df

train_df = add_cyclical_features(train_df)
test_df = add_cyclical_features(test_df)

print('Cyclical encoding added: hour_sin/cos, dow_sin/cos, month_sin/cos')
print(f'  hour_sin range: [{train_df["hour_sin"].min():.3f}, {train_df["hour_sin"].max():.3f}]')
print(f'  month_cos range: [{train_df["month_cos"].min():.3f}, {train_df["month_cos"].max():.3f}]')

Cyclical encoding added: hour_sin/cos, dow_sin/cos, month_sin/cos
  hour_sin range: [-1.000, 1.000]
  month_cos range: [-1.000, 1.000]


### T1.4 Coordinate Gridding + K-Fold Target Encoding ⭐⭐⭐

The most critical feature. Steps:
1. Discretize coordinates into grid cells (~100m resolution)
2. K-Fold Target Encode grid_id on train (prevents leakage)
3. Full-train stats encode grid_id on test

Expected: grid_te Spearman ρ with target ≈ 0.3–0.5. If >0.7 → leakage alarm.

In [6]:
# === T1.4a: Coordinate Gridding ===
# EDA showed coordinates are scaled to ~0.995-1.000 range (city-level)
# longitude span ~0.002, latitude span ~0.005
# grid_size=0.00005 yields ~40 lon x ~100 lat = ~4000 grids
# With 6M rows, that's ~1500 samples per grid on average — good balance
GRID_SIZE = 0.00005

def add_grid_features(df, grid_size):
    """Discretize scaled coordinates into grid cells."""
    df['grid_lon'] = (df['longitude_scaled'] / grid_size).astype('int32')
    df['grid_lat'] = (df['latitude_scaled'] / grid_size).astype('int32')
    # Combine into a single grid_id (integer key, faster than string)
    df['grid_id'] = df['grid_lon'] * 100000 + df['grid_lat']
    return df

train_df = add_grid_features(train_df, GRID_SIZE)
test_df = add_grid_features(test_df, GRID_SIZE)

n_grids_train = train_df['grid_id'].nunique()
n_grids_test = test_df['grid_id'].nunique()
unseen_grids = set(test_df['grid_id'].unique()) - set(train_df['grid_id'].unique())

print(f'Grid size: {GRID_SIZE}')
print(f'Unique grids — train: {n_grids_train:,}, test: {n_grids_test:,}')
print(f'Unseen grids in test: {len(unseen_grids):,} ({len(unseen_grids)/n_grids_test*100:.1f}%)')
print(f'Avg samples per grid: {len(train_df)/n_grids_train:,.0f}')

Grid size: 5e-05
Unique grids — train: 742, test: 584
Unseen grids in test: 21 (3.6%)
Avg samples per grid: 8,189


In [7]:
# === T1.4b: K-Fold Target Encoding Function ===
def kfold_target_encode(train_df, test_df, col, target='invalid_ratio',
                        n_splits=5, smooth=30):
    """
    K-Fold Target Encoding with Bayesian smoothing.
    
    - Train: each fold is encoded using stats from the other folds only
    - Test: encoded using full training set stats
    - Smoothing shrinks small-sample categories toward the global mean
    
    Returns: (train_encoded_array, test_encoded_series, encoding_map_dict)
    """
    global_mean = float(train_df[target].mean())
    # Use numpy array to avoid pandas float32/64 dtype conflicts
    train_encoded = np.full(len(train_df), np.nan, dtype='float64')
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    
    for fold_train_idx, fold_val_idx in kf.split(train_df):
        # Compute stats from training fold only
        fold_data = train_df.iloc[fold_train_idx]
        fold_stats = fold_data.groupby(col)[target].agg(['mean', 'count'])
        # Bayesian smoothing: shrink toward global mean when sample is small
        smoothed = (
            fold_stats['count'] * fold_stats['mean'] + smooth * global_mean
        ) / (fold_stats['count'] + smooth)
        # Encode the validation fold using numpy indexing
        mapped_vals = train_df.iloc[fold_val_idx][col].map(smoothed).values
        train_encoded[fold_val_idx] = mapped_vals
    
    # Fill unmapped values (rare edge case)
    nan_mask = np.isnan(train_encoded)
    train_encoded[nan_mask] = global_mean
    
    # Test set: use full training set stats
    full_stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    full_smoothed = (
        full_stats['count'] * full_stats['mean'] + smooth * global_mean
    ) / (full_stats['count'] + smooth)
    test_encoded = test_df[col].map(full_smoothed).fillna(global_mean).values
    
    # Save the encoding map for reproducibility
    encoding_map = full_smoothed.to_dict()
    
    return train_encoded.astype('float32'), test_encoded.astype('float32'), encoding_map

print('kfold_target_encode function defined')

kfold_target_encode function defined


In [8]:
# === T1.4c: Apply Grid Target Encoding ===
print('Running K-Fold Target Encoding on grid_id (this may take 1-2 minutes)...')

grid_te_train, grid_te_test, grid_encoding = kfold_target_encode(
    train_df, test_df, 'grid_id', 'invalid_ratio', n_splits=5, smooth=30
)
train_df['grid_te'] = grid_te_train
test_df['grid_te'] = grid_te_test

# Validate: check Spearman correlation with target
grid_te_spearman, _ = stats.spearmanr(train_df['grid_te'], train_df['invalid_ratio'])
print(f'grid_te Spearman rho with target: {grid_te_spearman:.4f}')

if grid_te_spearman > 0.7:
    print('WARNING: rho > 0.7 — potential data leakage!')
elif grid_te_spearman < 0.1:
    print('WARNING: rho < 0.1 — grid_size may be too large or implementation bug')
else:
    print('Validation PASSED: correlation is in expected range [0.1, 0.7]')

print(f'grid_te stats — mean: {train_df["grid_te"].mean():.4f}, '
      f'std: {train_df["grid_te"].std():.4f}, '
      f'range: [{train_df["grid_te"].min():.4f}, {train_df["grid_te"].max():.4f}]')

Running K-Fold Target Encoding on grid_id (this may take 1-2 minutes)...


grid_te Spearman rho with target: 0.3072
Validation PASSED: correlation is in expected range [0.1, 0.7]
grid_te stats — mean: 0.5020, std: 0.1166, range: [0.1831, 0.9555]


### T1.5 Tier 1 Checkpoint

Save intermediate results so we can resume from here if needed.

In [9]:
# === T1.5: Save Tier 1 Checkpoint ===
global_mean = float(train_df['invalid_ratio'].mean())

encoding_maps = {
    'grid': grid_encoding,
    'forcevent_median': forcevent_median,
    'grid_size': GRID_SIZE,
    'global_mean': global_mean,
}

with open(f'{OUTPUT_DIR}encoding_maps_tier1.pkl', 'wb') as f:
    pickle.dump(encoding_maps, f)

train_df.to_parquet(f'{OUTPUT_DIR}train_features_tier1.parquet', index=False)
test_df.to_parquet(f'{OUTPUT_DIR}test_features_tier1.parquet', index=False)

gc.collect()

print(f'Tier 1 checkpoint saved to {OUTPUT_DIR}')
print(f'  Train features: {train_df.shape[1]} columns')
print(f'  Test features:  {test_df.shape[1]} columns')
print(f'  Columns: {list(train_df.columns)}')

Tier 1 checkpoint saved to ../data/
  Train features: 23 columns
  Test features:  22 columns
  Columns: ['total_count', 'longitude_scaled', 'latitude_scaled', 'Precipitations', 'HauteurNeige', 'Temperature', 'ForceVent', 'day_of_week', 'month_of_year', 'hour', 'invalid_ratio', 'log_total_count', 'count_bin', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'grid_lon', 'grid_lat', 'grid_id', 'grid_te']


---
## Tier 2: Should-Do Features

Additional features to further improve model performance.

### T2.1 Time Period Binning

Based on EDA temporal patterns and Paper 4's commute/lunch peak findings,  
bin hours into meaningful time periods.

In [10]:
# === T2.1: Time Period Binning ===
# Vectorized with np.select (much faster than apply)
def add_time_period(df):
    """Bin hours into semantically meaningful time periods."""
    conditions = [
        df['hour'] < 7,   # early morning (rare in data)
        df['hour'] < 9,   # morning rush
        df['hour'] < 12,  # late morning
        df['hour'] < 14,  # lunch peak
        df['hour'] < 17,  # afternoon
        df['hour'] >= 17  # evening
    ]
    choices = [0, 1, 2, 3, 4, 5]
    df['time_period'] = np.select(conditions, choices, default=5).astype('int8')
    return df

train_df = add_time_period(train_df)
test_df = add_time_period(test_df)

print('Time period distribution (train):')
period_labels = {0: 'early_morn', 1: 'morning_rush', 2: 'late_morning',
                 3: 'lunch_peak', 4: 'afternoon', 5: 'evening'}
for period, label in period_labels.items():
    count = (train_df['time_period'] == period).sum()
    mean_vr = train_df.loc[train_df['time_period'] == period, 'invalid_ratio'].mean()
    print(f'  {period} ({label:15s}): {count:>10,} samples, mean VR = {mean_vr:.4f}')

Time period distribution (train):
  0 (early_morn     ):        525 samples, mean VR = 0.6339


  1 (morning_rush   ):    923,959 samples, mean VR = 0.5052
  2 (late_morning   ):  1,748,281 samples, mean VR = 0.5039
  3 (lunch_peak     ):  1,051,851 samples, mean VR = 0.4815
  4 (afternoon      ):  1,658,118 samples, mean VR = 0.4977
  5 (evening        ):    693,812 samples, mean VR = 0.5376


### T2.2 Grid × Time Period Cross Target Encoding

The same area may have different violation rates at different times of day.  
Use a combined key `grid_id × time_period` with stronger smoothing (fewer samples per group).

In [11]:
# === T2.2: Grid x Time Period Cross Target Encoding ===
# Create combined key
train_df['grid_period'] = (train_df['grid_id'].astype('int64') * 10 + train_df['time_period'].astype('int64'))
test_df['grid_period'] = (test_df['grid_id'].astype('int64') * 10 + test_df['time_period'].astype('int64'))

print('Running K-Fold TE on grid_period (this may take 2-3 minutes)...')

gp_te_train, gp_te_test, grid_period_encoding = kfold_target_encode(
    train_df, test_df, 'grid_period', 'invalid_ratio',
    n_splits=5, smooth=50  # stronger smoothing for finer-grained groups
)
train_df['grid_period_te'] = gp_te_train
test_df['grid_period_te'] = gp_te_test

# Validate
gp_te_spearman, _ = stats.spearmanr(train_df['grid_period_te'], train_df['invalid_ratio'])
print(f'grid_period_te Spearman rho with target: {gp_te_spearman:.4f}')

if gp_te_spearman > 0.7:
    print('WARNING: rho > 0.7 — potential leakage!')
else:
    print('Validation PASSED')

Running K-Fold TE on grid_period (this may take 2-3 minutes)...


grid_period_te Spearman rho with target: 0.3112
Validation PASSED


### T2.3 Weather Binary Features

Individual weather variables have weak correlation with the target,  
but binary indicators (raining / snowing) may capture threshold effects.

In [12]:
# === T2.3: Weather Binary Features ===
train_df['is_raining'] = (train_df['Precipitations'] > 0).astype('int8')
test_df['is_raining'] = (test_df['Precipitations'] > 0).astype('int8')

train_df['has_snow'] = (train_df['HauteurNeige'] > 0).astype('int8')
test_df['has_snow'] = (test_df['HauteurNeige'] > 0).astype('int8')

# Check whether binary weather features show any signal
rain_vr = train_df.groupby('is_raining')['invalid_ratio'].mean()
snow_vr = train_df.groupby('has_snow')['invalid_ratio'].mean()

print('Weather binary feature signals:')
print(f'  is_raining=0: VR={rain_vr[0]:.4f}, is_raining=1: VR={rain_vr[1]:.4f} (diff={rain_vr[1]-rain_vr[0]:.4f})')
print(f'  has_snow=0:   VR={snow_vr[0]:.4f}, has_snow=1:   VR={snow_vr[1]:.4f} (diff={snow_vr[1]-snow_vr[0]:.4f})')

Weather binary feature signals:
  is_raining=0: VR=0.5023, is_raining=1: VR=0.5028 (diff=0.0004)
  has_snow=0:   VR=0.5024, has_snow=1:   VR=0.5031 (diff=0.0007)


### T2.4 Grid-Level Statistics

Area-level aggregated features (computed from training set only):  
- `grid_avg_count`: average inspection count per area (captures enforcement intensity)  
- `grid_violation_std`: violation rate volatility per area  
- `grid_sample_count`: how many observations exist for the area (data richness)

In [13]:
# === T2.4: Grid-Level Statistics ===
# Compute from training set only
grid_stats = train_df.groupby('grid_id').agg(
    grid_avg_count=('total_count', 'mean'),
    grid_violation_std=('invalid_ratio', 'std'),
    grid_sample_count=('invalid_ratio', 'count')
).astype('float32')

# Fill NaN std (grids with only 1 sample)
grid_stats['grid_violation_std'] = grid_stats['grid_violation_std'].fillna(0)

# Merge onto train and test
train_df = train_df.merge(grid_stats, on='grid_id', how='left')
test_df = test_df.merge(grid_stats, on='grid_id', how='left')

# Fill test rows with unseen grid_ids using global stats
global_avg_count = float(train_df['total_count'].mean())
global_violation_std = float(train_df['invalid_ratio'].std())
global_sample_count = float(train_df.groupby('grid_id')['invalid_ratio'].count().median())

for col, fill_val in [('grid_avg_count', global_avg_count),
                       ('grid_violation_std', global_violation_std),
                       ('grid_sample_count', global_sample_count)]:
    test_df[col] = test_df[col].fillna(fill_val).astype('float32')

print('Grid statistics added:')
for col in ['grid_avg_count', 'grid_violation_std', 'grid_sample_count']:
    rho, _ = stats.spearmanr(train_df[col], train_df['invalid_ratio'])
    print(f'  {col}: Spearman rho = {rho:.4f}')

gc.collect()

Grid statistics added:


  grid_avg_count: Spearman rho = -0.1686


  grid_violation_std: Spearman rho = 0.0636


  grid_sample_count: Spearman rho = -0.1578


0

### T2.5 Save Tier 2 Checkpoint

In [14]:
# === T2.5: Save Tier 2 Checkpoint ===
# Update encoding maps
encoding_maps['grid_period'] = grid_period_encoding
encoding_maps['grid_stats'] = grid_stats.to_dict()
encoding_maps['global_avg_count'] = global_avg_count
encoding_maps['global_violation_std'] = global_violation_std
encoding_maps['global_sample_count'] = global_sample_count

with open(f'{OUTPUT_DIR}encoding_maps_tier2.pkl', 'wb') as f:
    pickle.dump(encoding_maps, f)

train_df.to_parquet(f'{OUTPUT_DIR}train_features_tier2.parquet', index=False)
test_df.to_parquet(f'{OUTPUT_DIR}test_features_tier2.parquet', index=False)

gc.collect()

print(f'Tier 2 checkpoint saved to {OUTPUT_DIR}')
print(f'  Train features: {train_df.shape[1]} columns')
print(f'  Test features:  {test_df.shape[1]} columns')

Tier 2 checkpoint saved to ../data/
  Train features: 31 columns
  Test features:  30 columns


---
## Feature Summary & Validation

In [15]:
# === Final Feature Overview ===
# Identify which columns are features (exclude target and intermediate keys)
exclude_cols = ['invalid_ratio', 'grid_lon', 'grid_lat', 'grid_id', 'grid_period']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

print(f'Total feature columns: {len(feature_cols)}')
print(f'Columns excluded (intermediate / target): {exclude_cols}')
print(f'\nFeature list:')
for i, col in enumerate(feature_cols, 1):
    dtype = train_df[col].dtype
    print(f'  {i:2d}. {col:25s} ({dtype})')

Total feature columns: 26
Columns excluded (intermediate / target): ['invalid_ratio', 'grid_lon', 'grid_lat', 'grid_id', 'grid_period']

Feature list:
   1. total_count               (int32)
   2. longitude_scaled          (float32)
   3. latitude_scaled           (float32)
   4. Precipitations            (float32)
   5. HauteurNeige              (float32)
   6. Temperature               (float32)
   7. ForceVent                 (float32)
   8. day_of_week               (int8)
   9. month_of_year             (int8)
  10. hour                      (int8)
  11. log_total_count           (float32)
  12. count_bin                 (int8)
  13. hour_sin                  (float32)
  14. hour_cos                  (float32)
  15. dow_sin                   (float32)
  16. dow_cos                   (float32)
  17. month_sin                 (float32)
  18. month_cos                 (float32)
  19. grid_te                   (float32)
  20. time_period               (int8)
  21. grid_period_te      

In [16]:
# === Spearman Correlation of All Features with Target ===
print('Spearman correlation with invalid_ratio (sorted by |rho|):')
print('-' * 55)

correlations = {}
for col in feature_cols:
    rho, pval = stats.spearmanr(train_df[col], train_df['invalid_ratio'])
    correlations[col] = rho

# Sort by absolute value
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

for col, rho in sorted_corr:
    marker = ' ***' if abs(rho) > 0.1 else (' **' if abs(rho) > 0.05 else '')
    print(f'  {col:25s}  rho = {rho:+.4f}{marker}')

Spearman correlation with invalid_ratio (sorted by |rho|):
-------------------------------------------------------


  grid_period_te             rho = +0.3112 ***
  grid_te                    rho = +0.3072 ***
  total_count                rho = -0.2953 ***
  log_total_count            rho = -0.2953 ***
  count_bin                  rho = -0.2886 ***
  grid_avg_count             rho = -0.1686 ***
  grid_sample_count          rho = -0.1578 ***
  month_of_year              rho = -0.0886 **
  latitude_scaled            rho = +0.0802 **
  month_sin                  rho = +0.0642 **
  grid_violation_std         rho = +0.0636 **
  longitude_scaled           rho = +0.0609 **
  hour_cos                   rho = +0.0357
  month_cos                  rho = +0.0269
  Temperature                rho = +0.0257
  hour                       rho = +0.0097
  hour_sin                   rho = -0.0096
  time_period                rho = +0.0095
  ForceVent                  rho = -0.0094
  dow_cos                    rho = +0.0050
  day_of_week                rho = +0.0034
  dow_sin                    rho = -0.0015
  Precipita

In [17]:
# === Final Validation Checks ===
print('=== Validation Checks ===')

# 1. No NaN remaining
train_nan = train_df[feature_cols].isnull().sum().sum()
test_feature_cols = [c for c in feature_cols if c in test_df.columns]
test_nan = test_df[test_feature_cols].isnull().sum().sum()
print(f'1. NaN check — train: {train_nan}, test: {test_nan}', 
      '  PASS' if train_nan == 0 and test_nan == 0 else '  FAIL!')

# 2. Train/test column consistency (test should have all features except target)
missing_in_test = [c for c in test_feature_cols if c not in test_df.columns]
print(f'2. Column consistency — missing in test: {missing_in_test}',
      '  PASS' if len(missing_in_test) == 0 else '  FAIL!')

# 3. Grid TE leakage check
grid_te_rho = correlations.get('grid_te', 0)
print(f'3. Grid TE leakage check — rho = {grid_te_rho:.4f}',
      '  PASS' if 0.1 < abs(grid_te_rho) < 0.7 else '  CHECK!')

# 4. Row counts unchanged
print(f'4. Row counts — train: {len(train_df):,}, test: {len(test_df):,}',
      '  PASS' if len(train_df) == 6_076_546 else '  FAIL!')

# 5. Memory usage
train_mem = train_df.memory_usage(deep=True).sum() / 1024**2
test_mem = test_df.memory_usage(deep=True).sum() / 1024**2
print(f'5. Memory — train: {train_mem:.0f} MB, test: {test_mem:.0f} MB')

print('\n=== Feature Engineering Complete ===')

=== Validation Checks ===
1. NaN check — train: 0, test: 0   PASS
2. Column consistency — missing in test: []   PASS
3. Grid TE leakage check — rho = 0.3072   PASS
4. Row counts — train: 6,076,546, test: 2,028,750   PASS
5. Memory — train: 620 MB, test: 199 MB

=== Feature Engineering Complete ===
